04_stage2_models

In [ ]:
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV, cross_validate, cross_val_predict
from sklearn.metrics import (make_scorer, precision_score, recall_score, f1_score, roc_auc_score,
                             average_precision_score, accuracy_score, brier_score_loss,
                             confusion_matrix, ConfusionMatrixDisplay, classification_report)
from sklearn.inspection import permutation_importance

SEED = 42
FAST = False
OUT_DIR = Path("outputs")

stage2 = pd.read_pickle(OUT_DIR / "stage2_working_women.pkl")
fl = pd.read_pickle(OUT_DIR / "feature_lists.pkl")
CAT, NUM = list(fl["FEATURES_CAT"]), list(fl["FEATURES_NUM"])
FEATURES = CAT + NUM

# Only features that non-participants also have (needed for scoring them in step 05)
assert not set(FEATURES) & {"usual_status", "emp_st", "economy_type", "own_economy_type", "Total_Income",
                            "Inc_Cat", "Individual_Precarity", "Social_Security", "b5pt1q11_perv1"}
print(stage2.shape)
for t in ["y_gig_vs_trad", "y_gig_vs_other", "y_gigstartup_vs_trad"]:
    v = stage2[t].dropna()
    print(f"{t}: n={len(v):,}, positive share={v.mean():.3f}")

In [ ]:
def make_pipe(model):
    pre = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=30), CAT),
        ("num", StandardScaler(), NUM)])
    return Pipeline([("pre", pre), ("model", model)])

def metrics(y_true, proba, thr=0.5):
    pred = (proba >= thr).astype(int)
    return {"accuracy": accuracy_score(y_true, pred),
            "precision": precision_score(y_true, pred, zero_division=0),
            "recall": recall_score(y_true, pred), "f1": f1_score(y_true, pred),
            "roc_auc": roc_auc_score(y_true, proba), "pr_auc": average_precision_score(y_true, proba),
            "brier": brier_score_loss(y_true, proba)}

scoring = {"accuracy": "accuracy", "precision": make_scorer(precision_score, zero_division=0),
           "recall": "recall", "f1": "f1", "roc_auc": "roc_auc", "pr_auc": "average_precision"}

def run_stage2(target, title):
    d = stage2[stage2[target].notna()].reset_index(drop=True)
    y, g = d[target].astype(int), d["HHID"]
    tri, tei = next(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED).split(d, y, g))
    Xtr, ytr, gtr = d.loc[tri, FEATURES], y.iloc[tri], g.iloc[tri]
    Xte, yte = d.loc[tei, FEATURES], y.iloc[tei]
    assert set(g.iloc[tri]).isdisjoint(set(g.iloc[tei])), "household leakage"
    prev = ytr.mean()
    print(f"\n===== {title} =====\nTrain {len(Xtr):,} | Test {len(Xte):,} | positive share train {prev:.3f}, test {yte.mean():.3f}")
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)

    gs_dt = GridSearchCV(
        make_pipe(DecisionTreeClassifier(class_weight="balanced", random_state=SEED)),
        {"model__max_depth": [3, 5] if FAST else [3, 4, 6, 8],
         "model__min_samples_leaf": [50] if FAST else [20, 50, 100]},
        scoring="average_precision", cv=cv, n_jobs=-1).fit(Xtr, ytr, groups=gtr)
    print("Best tree:", gs_dt.best_params_, "| CV PR-AUC %.3f" % gs_dt.best_score_)

    gs_rf = GridSearchCV(
        make_pipe(RandomForestClassifier(n_estimators=60 if FAST else 150, max_features="sqrt",
                                         class_weight="balanced_subsample", n_jobs=-1, random_state=SEED)),
        {"model__max_depth": [8] if FAST else [8, 12, 16],
         "model__min_samples_leaf": [30] if FAST else [20, 50]},
        scoring="average_precision", cv=cv, n_jobs=1, verbose=1).fit(Xtr, ytr, groups=gtr)
    print("Best forest:", gs_rf.best_params_, "| CV PR-AUC %.3f" % gs_rf.best_score_)

    models = {"Majority baseline": make_pipe(DummyClassifier(strategy="most_frequent")),
              "Logistic regression": make_pipe(LogisticRegression(max_iter=1000, class_weight="balanced")),
              "Decision tree": gs_dt.best_estimator_, "Random forest": gs_rf.best_estimator_}

    rows = []
    for name, pipe in models.items():
        r = cross_validate(clone(pipe), Xtr, ytr, groups=gtr, cv=cv, scoring=scoring)
        rows.append({"model": name, **{m: r[f"test_{m}"].mean() for m in scoring}})
    cv_table = pd.DataFrame(rows).round(3)
    best = (cv_table[cv_table["model"] != "Majority baseline"]
            .sort_values(["pr_auc", "f1"], ascending=False).iloc[0]["model"])

    oof = cross_val_predict(clone(models[best]), Xtr, ytr, groups=gtr, cv=cv, method="predict_proba")[:, 1]
    grid = np.linspace(0.05, 0.95, 91)
    thr = grid[int(np.argmax([f1_score(ytr, oof >= t) for t in grid]))]

    fitted, trows = {}, []
    for name, pipe in models.items():
        fitted[name] = clone(pipe).fit(Xtr, ytr)
        trows.append({"model": name, **metrics(yte, fitted[name].predict_proba(Xte)[:, 1])})
    test_table = pd.DataFrame(trows).round(3)
    test_table["pr_auc_lift_vs_prevalence"] = (test_table["pr_auc"] / yte.mean()).round(2)

    return dict(title=title, target=target, d=d, Xtr=Xtr, ytr=ytr, Xte=Xte, yte=yte, models=models,
                fitted=fitted, cv_table=cv_table, test_table=test_table, best=best, thr=thr,
                p_te=fitted[best].predict_proba(Xte)[:, 1])

In [ ]:
main = run_stage2("y_gig_vs_trad", "MAIN: Gig-proxy vs Traditional (startup set aside)")
print("\nCross-validated results:"); display(main["cv_table"])
print("\nTest-set results (PR-AUC lift = PR-AUC divided by the share of gig-proxy women):"); display(main["test_table"])
main["cv_table"].to_csv(OUT_DIR / "stage2_main_cv.csv", index=False)
main["test_table"].to_csv(OUT_DIR / "stage2_main_test.csv", index=False)

best, thr, yte, p_te = main["best"], main["thr"], main["yte"], main["p_te"]
print(f"\nBest model: {best} | tuned threshold {thr:.2f}")
print("At 0.5  :", {k: round(v, 3) for k, v in metrics(yte, p_te).items()})
print("At tuned:", {k: round(v, 3) for k, v in metrics(yte, p_te, thr).items()})
print("\nPer-class report at tuned threshold:")
print(classification_report(yte, p_te >= thr, target_names=["Traditional", "Gig-proxy"], digits=3))

ConfusionMatrixDisplay(confusion_matrix(yte, p_te >= thr),
                       display_labels=["Traditional", "Gig-proxy"]).plot(cmap="Blues")
plt.title(f"Stage 2 - {best} (threshold {thr:.2f})")
plt.savefig(OUT_DIR / "fig_stage2_confusion.png", dpi=150); plt.show()

In [ ]:
Xtr, ytr, Xte = main["Xtr"], main["ytr"], main["Xte"]

lr = clone(main["models"]["Logistic regression"]).fit(Xtr, ytr)
names = [n.replace("cat__", "").replace("num__", "") for n in lr.named_steps["pre"].get_feature_names_out()]
coef = pd.Series(lr.named_steps["model"].coef_[0], index=names)
odds = pd.DataFrame({"coef": coef, "odds_ratio": np.exp(coef)}).round(3).sort_values("coef")
print("Associated with MORE gig-proxy work:"); display(odds.tail(12).iloc[::-1])
print("Associated with LESS gig-proxy work (more traditional):"); display(odds.head(12))
odds.to_csv(OUT_DIR / "stage2_odds_ratios.csv")

samp = Xte.sample(min(8000, len(Xte)), random_state=SEED)
ysamp = main["yte"].loc[samp.index]
pi = permutation_importance(main["fitted"][main["best"]], samp, ysamp, scoring="average_precision",
                            n_repeats=3 if FAST else 5, random_state=SEED)
imp = pd.Series(pi.importances_mean, index=FEATURES).sort_values()
imp.plot.barh(figsize=(7, 6), color="#1F3A5F")
plt.xlabel("Drop in PR-AUC"); plt.title(f"Stage 2 permutation importance - {main['best']}")
plt.tight_layout(); plt.savefig(OUT_DIR / "fig_stage2_importance.png", dpi=150); plt.show()
imp.sort_values(ascending=False).round(4).to_csv(OUT_DIR / "stage2_importance.csv")

In [ ]:
alt_other = run_stage2("y_gig_vs_other", "ROBUSTNESS: Gig-proxy vs all other workers")
alt_gs = run_stage2("y_gigstartup_vs_trad", "SENSITIVITY: Gig + Startup vs Traditional")

summary = []
for r in [main, alt_other, alt_gs]:
    row = r["test_table"][r["test_table"]["model"] == r["best"]].iloc[0].to_dict()
    row.update({"target": r["title"], "positive_share_test": round(float(r["yte"].mean()), 3),
                "tuned_threshold": round(float(r["thr"]), 2)})
    summary.append(row)
summary = pd.DataFrame(summary)[["target", "model", "positive_share_test", "tuned_threshold",
                                 "precision", "recall", "f1", "roc_auc", "pr_auc", "pr_auc_lift_vs_prevalence", "brier"]]
display(summary); summary.to_csv(OUT_DIR / "stage2_target_comparison.csv", index=False)

In [ ]:
# Unfitted pipelines with tuned settings; step 05 refits them on all working women and calibrates them
joblib.dump({
    "main":  {"name": main["best"],      "pipe": clone(main["models"][main["best"]]),
              "threshold": float(main["thr"])},
    "other": {"name": alt_other["best"], "pipe": clone(alt_other["models"][alt_other["best"]]),
              "threshold": float(alt_other["thr"])},
    "features": FEATURES, "cat": CAT, "num": NUM,
}, OUT_DIR / "stage2_model_specs.joblib")
print("Saved outputs/stage2_model_specs.joblib")